# ReAct Agent with LangGraph

### Objectives:
1. **Learn How to Create Tools in LangGraph**: Use the `@tool` decorator to build custom, typed Python tools with docstrings.
2. **How to Create a ReAct Graph**: Structure the Reason + Act loop (`Agent Node` $\leftrightarrow$ `Tool Node`) with conditional routing.
3. **Work with Different Message Types (including `ToolMessage`)**: Understand `HumanMessage`, `AIMessage` (with `tool_calls`), and `ToolMessage` (with execution output and `tool_call_id`).
4. **Test Robustness of Our Graph**: Handle tool errors, multi-step queries, inputs requiring multiple tool calls, and queries that need no tools.

**Main Goal:** Build a robust, end-to-end ReAct (Reasoning + Acting) Agent from scratch using LangGraph.

## 0. Imports & Setup

In [ ]:
# Install dependencies if needed:
# !pip install langgraph langchain-core langchain-openai python-dotenv

In [ ]:
import os
import json
from typing import Annotated, Sequence, TypedDict, Literal
from dotenv import load_dotenv

# LangChain Tools and Messages
from langchain_core.tools import tool
from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    AIMessage,
    SystemMessage,
    ToolMessage
)
from langchain_openai import ChatOpenAI

# LangGraph Components
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

load_dotenv()

## 1. Objective 1: Creating Custom Tools

We use LangChain's `@tool` decorator. Docstrings and type hints are vital because LLMs read them to decide when and how to call each tool.

In [ ]:
@tool
def calculate(expression: str) -> str:
    """Useful for evaluating mathematical expressions like '25 * 4 + 10' or 'sqrt(144)'."""
    import math
    safe_dict = {"__builtins__": None, "math": math, "abs": abs, "round": round, "sum": sum, "pow": pow}
    try:
        result = eval(expression, safe_dict, {})
        return f"Result: {result}"
    except Exception as e:
        return f"Error calculating expression: {str(e)}"

@tool
def get_weather(city: str) -> str:
    """Useful for getting current weather information for a given city."""
    mock_weather = {
        "new york": "Sunny, 22°C with light breeze",
        "london": "Cloudy with intermittent rain, 15°C",
        "tokyo": "Clear skies, 19°C",
        "paris": "Mild, 18°C with moderate humidity"
    }
    normalized_city = city.strip().lower()
    return mock_weather.get(normalized_city, f"Weather data for '{city}' is currently unavailable (Sunny, 20°C assumed).")

@tool
def lookup_product_price(product_name: str) -> str:
    """Lookup the inventory price for a specific product name."""
    catalog = {
        "laptop": 1200,
        "headphones": 150,
        "mouse": 40,
        "keyboard": 90,
        "monitor": 300
    }
    normalized = product_name.strip().lower()
    if normalized in catalog:
        return f"The price of {product_name} is ${catalog[normalized]}."
    return f"Product '{product_name}' not found in catalog."

# Combine tools list
tools = [calculate, get_weather, lookup_product_price]
print(f"Created {len(tools)} tools: {[t.name for t in tools]}")

## 2. Objective 2: Defining the State and Binding Tools to the LLM

We bind our tools to `gpt-4o` using `llm.bind_tools(tools)`. This teaches the LLM how to trigger tool calls with valid JSON arguments.

In [ ]:
# 1. Define State with add_messages reducer
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

# 2. Initialize LLM
llm = ChatOpenAI(model="gpt-4o", temperature=0.0)

# 3. Bind tools to the model
llm_with_tools = llm.bind_tools(tools)
print("LLM bound with tools successfully!")

## 3. Objective 3: Messages and ReAct Node Architecture

The ReAct cycle flows between two key nodes:
1. **`agent_node`**: Calls the LLM. If the LLM determines a tool is needed, it responds with an `AIMessage` containing `tool_calls`.
2. **`tool_node`**: Executes requested tools and returns `ToolMessage` objects linking the result back via `tool_call_id`.
3. **`tools_condition`**: Directs flow:
   - If `tool_calls` exist $\rightarrow$ proceed to `tools`.
   - If no `tool_calls` $\rightarrow$ answer is complete, route to `END`.

In [ ]:
SYSTEM_PROMPT = SystemMessage(
    content="You are an expert ReAct assistant with access to math, weather, and price lookup tools. "
            "Always use tools when precise facts or calculations are needed."
)

def agent_node(state: AgentState) -> dict:
    """Calls the model with tools enabled and appends response."""
    messages = list(state["messages"])
    if not messages or not isinstance(messages[0], SystemMessage):
        messages = [SYSTEM_PROMPT] + messages
        
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

# Prebuilt ToolNode executes any tools called by the AIMessage
tool_node = ToolNode(tools)

## 4. Building the ReAct Graph

In [ ]:
# Initialize graph builder
workflow = StateGraph(AgentState)

# Add Agent and Tool Nodes
workflow.add_node("agent", agent_node)
workflow.add_node("tools", tool_node)

# Start with agent reasoning
workflow.add_edge(START, "agent")

# Conditional Edge: Decide whether to execute tools or finish
workflow.add_conditional_edges(
    "agent",
    tools_condition,
    {
        "tools": "tools",
        END: END
    }
)

# Loop back: Tools return output to Agent for next reasoning step
workflow.add_edge("tools", "agent")

# Compile the ReAct Agent
react_app = workflow.compile()
print("ReAct Agent graph compiled successfully!")

## 5. Objective 4: Testing Robustness Across Scenarios

We test four different scenarios to ensure the graph is robust:
1. **No Tool Needed**: Simple conversational query.
2. **Single Tool Call**: Price or Weather lookup.
3. **Multi-Step Tool Chaining**: Look up two product prices, then calculate total cost with discount.
4. **Error Handling & Fallbacks**: Invalid expression or unknown query.

In [ ]:
def run_react_test(query: str, test_title: str):
    print(f"\n{'='*70}")
    print(f"TEST: {test_title}")
    print(f"Query: {query}")
    print(f"{'='*70}")
    
    try:
        result = react_app.invoke({"messages": [HumanMessage(content=query)]})
        print("\n--- Execution Steps & Messages ---")
        for msg in result["messages"]:
            if isinstance(msg, HumanMessage):
                print(f"[Human]: {msg.content}")
            elif isinstance(msg, AIMessage):
                if getattr(msg, "tool_calls", None):
                    for tc in msg.tool_calls:
                        print(f"[AI Tool Call]: {tc['name']} with args {tc['args']}")
                else:
                    print(f"[AI Final Answer]: {msg.content}")
            elif isinstance(msg, ToolMessage):
                print(f"[Tool Result ({msg.name})]: {msg.content}")
        print(f"{'='*70}\n")
    except Exception as e:
        print(f"Execution note (Live API key required): {e}")

In [ ]:
# Test 1: Direct Answer (No tool needed)
run_react_test("Hi, who are you and what tools do you have?", "Test 1: Conversational query without tool execution")

In [ ]:
# Test 2: Single Tool Call
run_react_test("What is the weather like in Paris right now?", "Test 2: Single Weather Tool Call")

In [ ]:
# Test 3: Multi-step Reasoning & Tool Chaining
run_react_test(
    "I want to buy 2 laptops and 3 mice. Look up their prices and compute my total with a 10% discount!",
    "Test 3: Multi-Step Tool Chaining (Product Lookup + Math Calculation)"
)

In [ ]:
# Test 4: Robustness with Missing Data
run_react_test("How much does a flying car cost?", "Test 4: Handling missing item gracefully")